# Chapter 15 — Deep Learning on Unseen Data: Introducing Federated Learning

**Nama:** Omar Al Farizy Hutagalung  
**NIM:** 101032300034  
**Mata Kuliah:** Deep Learning

## Pokok bahasan menurut buku
Bab ini membahas:
1. Masalah privasi dalam deep learning.
2. Federated learning.
3. Belajar mendeteksi spam.
4. Serangan terhadap federated learning melalui kebocoran pembaruan model.
5. Secure aggregation.
6. Homomorphic encryption.
7. Federated learning dengan pembaruan terenkripsi.

Notebook ini mengikuti urutan konsep bab, dengan contoh numerik kecil untuk pembelajaran. Dataset Enron asli dan framework autograd dari bab sebelumnya tidak disertakan, sehingga contoh di sini tidak mengklaim mereplikasi angka akurasi eksperimen buku.

## 1. Masalah privasi dalam deep learning

Model deep learning membutuhkan data pelatihan, tetapi data yang berguna sering berisi informasi sensitif, misalnya email, catatan kesehatan, atau aktivitas pengguna. Mengumpulkan semua data pada satu server dapat meningkatkan risiko privasi.

Gagasan utama *federated learning* adalah **membawa model ke data**, bukan selalu membawa seluruh data ke satu tempat. Data tetap berada di perangkat atau lingkungan lokal; peserta melatih model secara lokal lalu mengirim pembaruan model ke proses agregasi.

Penting: federated learning saja tidak otomatis menjamin privasi. Pembaruan model atau gradien masih dapat membocorkan informasi tentang contoh pelatihan.

## 2. Skema federated learning

Siklus sederhana:
1. Server mengirim parameter model awal ke peserta.
2. Setiap peserta melatih model menggunakan data lokalnya.
3. Peserta mengirim pembaruan parameter atau gradien.
4. Server mengagregasikan pembaruan, misalnya dengan rata-rata.
5. Model hasil agregasi dikirim kembali untuk putaran berikutnya.

Berikut simulasi angka kecil. Ini menunjukkan mekanisme agregasi, bukan model spam yang dilatih pada email.

In [1]:
import numpy as np

server_weights = np.array([0.2, -0.1, 0.4])
client_updates = [
    np.array([0.03, -0.02, 0.01]),
    np.array([0.01,  0.04, -0.02]),
    np.array([-0.02, 0.01, 0.03]),
]

print("Bobot awal server:", server_weights)
for i, update in enumerate(client_updates, start=1):
    print(f"Pembaruan peserta {i}:", update)

mean_update = np.mean(client_updates, axis=0)
new_weights = server_weights + mean_update
print("Rata-rata pembaruan:", mean_update)
print("Bobot setelah agregasi:", new_weights)

Bobot awal server: [ 0.2 -0.1  0.4]
Pembaruan peserta 1: [ 0.03 -0.02  0.01]
Pembaruan peserta 2: [ 0.01  0.04 -0.02]
Pembaruan peserta 3: [-0.02  0.01  0.03]
Rata-rata pembaruan: [0.00666667 0.01       0.00666667]
Bobot setelah agregasi: [ 0.20666667 -0.09        0.40666667]


## 3. Belajar mendeteksi spam

Buku menggunakan contoh klasifikasi email spam dan ham (bukan spam) berdasarkan korpus email Enron dan model embedding. Email diubah menjadi token/indeks sehingga dapat diproses oleh model. Contoh buku membentuk data pelatihan dan pengujian, lalu melatih model secara lokal.

Untuk menjaga notebook mandiri dan tidak memerlukan data eksternal, contoh berikut hanya memperlihatkan preprocessing teks yang sangat sederhana. Ini bukan model spam terlatih dan bukan reproduksi dataset Enron.

In [2]:
emails = [
    ("win money now", 1),       # 1 = spam (label ilustratif)
    ("meeting at noon", 0),     # 0 = ham
    ("claim your prize", 1),
    ("project update attached", 0),
]

tokenized = [message.lower().split() for message, _ in emails]
vocab = sorted({token for tokens in tokenized for token in tokens})
word_to_id = {word: idx for idx, word in enumerate(vocab)}

for (message, label), tokens in zip(emails, tokenized):
    ids = [word_to_id[token] for token in tokens]
    print(f"{message!r} -> token IDs {ids}; label={label}")

print("Vocabulary:", word_to_id)

'win money now' -> token IDs [10, 4, 6]; label=1
'meeting at noon' -> token IDs [3, 0, 5]; label=0
'claim your prize' -> token IDs [2, 11, 7]; label=1
'project update attached' -> token IDs [8, 9, 1]; label=0
Vocabulary: {'at': 0, 'attached': 1, 'claim': 2, 'meeting': 3, 'money': 4, 'noon': 5, 'now': 6, 'prize': 7, 'project': 8, 'update': 9, 'win': 10, 'your': 11}


## 4. Bagaimana pembaruan model dapat membocorkan data?

Dalam contoh konseptual buku, peserta Bob melatih model pada satu email. Jika pembaruan bobot yang dikirim Bob dapat diperiksa satu per satu, perubahan tertentu dapat mengungkap kata-kata yang muncul pada emailnya—termasuk informasi rahasia. Artinya, data tidak harus dikirim mentah untuk tetap menimbulkan risiko privasi.

Contoh berikut hanya mengilustrasikan prinsip kebocoran: sebuah vektor pembaruan memiliki perubahan pada posisi token tertentu. Ini bukan serangan nyata atau implementasi pemulihan data.

In [3]:
vocabulary = ["meeting", "project", "password", "pizza", "schedule"]
update = np.array([0.0, 0.0, 0.8, -0.6, 0.0])

changed_positions = np.flatnonzero(update != 0)
print("Indeks dengan perubahan:", changed_positions.tolist())
print("Token terkait perubahan:", [vocabulary[i] for i in changed_positions])
print("Peringatan: contoh ini memetakan posisi yang sudah diketahui; bukan metode umum untuk membongkar gradien.")

Indeks dengan perubahan: [2, 3]
Token terkait perubahan: ['password', 'pizza']
Peringatan: contoh ini memetakan posisi yang sudah diketahui; bukan metode umum untuk membongkar gradien.


## 5. Secure aggregation

*Secure aggregation* bertujuan agar agregator memperoleh gabungan pembaruan peserta tanpa melihat pembaruan individual secara langsung. Buku memperkenalkan ide *randomized response* dan *plausible deniability*: noise membuat hasil individual lebih sulit diatribusikan, sedangkan statistik agregat dapat tetap berguna.

Agregasi saja bukan jaminan privasi. Sistem nyata memerlukan protokol kriptografi yang dirancang dengan benar, pengelolaan peserta yang aman, serta pertimbangan trade-off antara privasi dan kualitas model.

In [4]:
# Demonstrasi agregasi biasa: server hanya membutuhkan jumlah/rata-rata.
# Dalam kode biasa ini, server masih dapat melihat tiap update sebelum menghitung rata-rata.
updates = np.array([
    [0.10, 0.00, -0.05],
    [0.00, 0.15, -0.02],
    [0.05, 0.05,  0.02],
])
aggregate_sum = updates.sum(axis=0)
aggregate_mean = updates.mean(axis=0)

print("Jumlah pembaruan:", aggregate_sum)
print("Rata-rata pembaruan:", aggregate_mean)
print("Catatan: ini hanya agregasi numerik biasa, bukan protokol secure aggregation.")

Jumlah pembaruan: [ 0.15  0.2  -0.05]
Rata-rata pembaruan: [ 0.05        0.06666667 -0.01666667]
Catatan: ini hanya agregasi numerik biasa, bukan protokol secure aggregation.


## 6. Homomorphic encryption

*Homomorphic encryption* memungkinkan operasi tertentu dilakukan pada nilai terenkripsi tanpa mendekripsinya terlebih dahulu. Buku mencontohkan skema Paillier melalui pustaka `phe`, yang mendukung penjumlahan ciphertext.

Secara konseptual, jika enkripsi mendukung operasi penjumlahan homomorfik, hasil penjumlahan ciphertext dapat didekripsi menjadi jumlah plaintext asal. Contoh berikut menggunakan `phe` jika pustaka tersedia. Ini demonstrasi edukatif, bukan protokol produksi.

In [5]:
try:
    import phe
except ImportError:
    print("Pustaka 'phe' belum terpasang.")
    print("Untuk mencoba contoh: pip install phe")
else:
    public_key, private_key = phe.generate_paillier_keypair(n_length=128)
    encrypted_x = public_key.encrypt(5)
    encrypted_y = public_key.encrypt(3)
    encrypted_sum = encrypted_x + encrypted_y
    decrypted_sum = private_key.decrypt(encrypted_sum)
    print("Dekripsi hasil penjumlahan terenkripsi:", decrypted_sum)
    assert decrypted_sum == 8

Pustaka 'phe' belum terpasang.
Untuk mencoba contoh: pip install phe


### Apa yang dilindungi dan apa yang tidak?

- Public key digunakan untuk mengenkripsi; private key diperlukan untuk mendekripsi.
- Operasi homomorfik yang didukung dapat diterapkan pada ciphertext.
- Enkripsi tidak otomatis mengatasi semua risiko: implementasi, pengelolaan kunci, metadata, dan protokol agregasi tetap penting.
- Contoh buku menggunakan panjang kunci kecil untuk demonstrasi; itu **bukan** rekomendasi keamanan produksi.

## 7. Federated learning dengan pembaruan terenkripsi

Alur konseptual yang dibahas di bab:
1. Pemilik model membagikan model dan public key kepada peserta.
2. Peserta melatih model pada data lokal.
3. Pembaruan yang perlu digabung dienkripsi.
4. Peserta/protokol menggabungkan ciphertext tanpa membuka nilai individual.
5. Hasil agregat didekripsi oleh pihak yang berwenang, lalu digunakan untuk memperbarui model.

Untuk implementasi nyata, diperlukan desain yang tepat tentang apa yang dienkripsi, bagaimana agregasi dilakukan, serta bagaimana bobot hasil agregat dihitung. Notebook ini tidak menyajikan protokol produksi dan tidak mengklaim menjalankan eksperimen federated learning lengkap.

## 8. Rangkuman dan pertanyaan refleksi

### Rangkuman
- Federated learning memungkinkan pelatihan dari data yang tetap berada di lokasi peserta.
- Pembaruan model individual tetap dapat mengungkap informasi sensitif.
- Secure aggregation bertujuan menyembunyikan pembaruan individual di balik agregat.
- Homomorphic encryption memungkinkan operasi tertentu dilakukan pada ciphertext.
- Privasi, utilitas model, keamanan implementasi, dan pengelolaan kunci perlu dipertimbangkan bersama.

### Pertanyaan refleksi
1. Apa perbedaan memindahkan data ke server dengan membawa model ke data?
2. Mengapa federated learning tidak otomatis menjamin privasi?
3. Bagaimana pembaruan model bisa mengungkap informasi tentang data lokal?
4. Apa tujuan secure aggregation?
5. Apa yang dimungkinkan oleh homomorphic encryption?
6. Mengapa contoh kunci pendek tidak cocok untuk sistem produksi?

**Batasan:** contoh spam menggunakan empat kalimat buatan, bukan korpus Enron; simulasi agregasi tidak menerapkan protokol secure aggregation; contoh enkripsi hanya menunjukkan penjumlahan homomorfik dasar.